# Hierarchical Architecture — LangGraph

**Pattern:** Supervisor node routes to workers  
**Framework:** LangGraph  

```
START → supervisor ─┬─→ research_weather → supervisor
                    ├─→ research_safety  → supervisor
                    ├─→ format_report    → supervisor
                    └─→ executive_summary → END
```

**LangGraph approach:** Supervisor is a node that reads state and decides what to do next.  
Each worker runs then returns to the supervisor — classic hub-and-spoke pattern.

In [ ]:
import os
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
print("✓ Ready")

In [ ]:
WEATHER_DATA = {"tokyo": "Clear, 18°C, 9/10", "paris": "Partly Cloudy, 16°C, 7/10", "bangalore": "Rainy, 25°C, 6/10"}
SAFETY_DATA  = {"tokyo": "Low, 10/10", "paris": "Low, 8/10", "bangalore": "Medium, 6/10"}

class HierarchyState(TypedDict):
    cities: list
    current_city_index: int
    weather_reports: list
    safety_reports: list
    formatted_sections: list
    executive_summary: Optional[str]
    next_step: str

print("State schema defined")

In [ ]:
def supervisor(state: HierarchyState) -> HierarchyState:
    idx = state["current_city_index"]
    n = len(state["cities"])
    if idx < n and len(state["weather_reports"]) <= idx:
        next_step = "research_weather"
    elif idx < n and len(state["safety_reports"]) <= idx:
        next_step = "research_safety"
    elif idx < n and len(state["formatted_sections"]) <= idx:
        next_step = "format_report"
    else:
        next_step = "executive_summary"
    print(f"  [supervisor] → {next_step} (city {idx}/'{
        state['cities'][idx] if idx < n else 'done'}'")
    return {"next_step": next_step}

def research_weather(state):
    idx = state["current_city_index"]; city = state["cities"][idx]
    r = llm.invoke([SystemMessage(content="1-sentence weather report."), HumanMessage(content=f"{city}: {WEATHER_DATA.get(city.lower(),'N/A')}")])
    return {"weather_reports": state["weather_reports"] + [r.content]}

def research_safety(state):
    idx = state["current_city_index"]; city = state["cities"][idx]
    r = llm.invoke([SystemMessage(content="1-sentence safety report."), HumanMessage(content=f"{city}: {SAFETY_DATA.get(city.lower(),'N/A')}")])
    return {"safety_reports": state["safety_reports"] + [r.content]}

def format_report(state):
    idx = state["current_city_index"]; city = state["cities"][idx]
    r = llm.invoke([SystemMessage(content="Format as '### City' markdown section."),
                    HumanMessage(content=f"{city}\nWeather: {state['weather_reports'][idx]}\nSafety: {state['safety_reports'][idx]}")])
    return {"formatted_sections": state["formatted_sections"] + [r.content], "current_city_index": idx + 1}

def executive_summary(state):
    r = llm.invoke([SystemMessage(content="Executive summary with top recommendation."),
                    HumanMessage(content="\n\n".join(state["formatted_sections"]))])
    return {"executive_summary": r.content}

print("Nodes defined: supervisor | weather | safety | format | summary")

In [ ]:
builder = StateGraph(HierarchyState)
for name, fn in [("supervisor", supervisor), ("research_weather", research_weather),
                 ("research_safety", research_safety), ("format_report", format_report),
                 ("executive_summary", executive_summary)]:
    builder.add_node(name, fn)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", lambda s: s["next_step"], {
    "research_weather": "research_weather", "research_safety": "research_safety",
    "format_report": "format_report", "executive_summary": "executive_summary",
})
for node in ["research_weather", "research_safety", "format_report"]:
    builder.add_edge(node, "supervisor")
builder.add_edge("executive_summary", END)

graph = builder.compile()
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
result = graph.invoke({
    "cities": ["Tokyo", "Paris"], "current_city_index": 0,
    "weather_reports": [], "safety_reports": [], "formatted_sections": [],
    "executive_summary": None, "next_step": "",
})

for section in result["formatted_sections"]:
    print(section); print()
print("## Executive Summary")
print(result["executive_summary"])

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Supervisor as a routing node | Hierarchy = supervisor reads state → routes to next worker |
| Workers return to supervisor | Hub-and-spoke: all roads lead back to supervisor |
| `conditional_edges` on supervisor | Supervisor's routing decision is an explicit graph edge |

### Next: [CrewAI Hierarchical →](../CrewAI/hierarchical.ipynb)